In [1]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [2]:
model_name = "facebook/nllb-200-1.3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype=torch.float16,  # FP16 instead of FP32 to fit in 8GB
).to(device)
print(f"Loaded {model_name}")

# Check actual GPU memory used
allocated_gb = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory used: {allocated_gb:.2f} GB")

Loaded facebook/nllb-200-1.3B
GPU memory used: 2.75 GB


In [3]:
test_sentences = [
    "مرحبا، كيف حالك؟",
    "اسمي محمد علي، ولدت في حلب عام 1987",
    "أسكن في شارع الرشيد رقم 42، دمشق",  # The R.R.R.D.M failure case
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    "رقم الهوية الشخصية: 0123456789",
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    "بناءً على المادة الثانية من قانون العقوبات",
    "السيد عبد الرحمن الحسيني",
    "إن شاء الله سأصل غدا في المساء",
]

print(f"NLLB-200-1.3B on {len(test_sentences)} test sentences\n")

start = time.time()
for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

elapsed = time.time() - start
print(f"Total: {elapsed:.2f}s ({elapsed/len(test_sentences):.2f}s per sentence)")

NLLB-200-1.3B on 10 test sentences

1. AR: مرحبا، كيف حالك؟
   DE: Hallo, wie geht's?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Muhammad Ali. Ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42-Rashid-Straße, Damaskus.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Die Nummer des Personenkennzeichens: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Nach Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Ich bin ein großer Freund von Mr. Abdul Rahman al-Husseini.

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Wenn Gott will, komme ich morgen Abend.

Total: 9.80s (0

In [4]:
peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak GPU memory during inference: {peak_gb:.2f} GB")

Peak GPU memory during inference: 2.76 GB
